# Confusion matrices
Generates aggregated confusion matrix figures across all LOSO sites or SKF folds. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `run_experiments.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy matplotlib

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from data_io.centralised_logger import CentralisedLogger
from utils.paths import get_project_root
from visualisation.common import discover_timestamps
from visualisation.confusion_matrices import default_title, default_filename, plot_confusion_matrix

## 3. Configuration **[CONFIGURE]**
The `RUNS` table maps a display label to a `(folder_name, tags)` pair. Each tag assigns a `'CV-Experiment'` label to a timestamp index discovered in that folder. Valid tag values are: `'SKF-Baseline'`, `'SKF-Optimised'`, `'LOSO-Baseline'`, `'LOSO-Optimised'`.

Use `FILTER` to restrict which runs are plotted. Set any entry to `None` to include everything.

| Column | Description |
|---|---|
| `folder_name` | The `model_name` string passed to `CentralisedLogger` |
| `tags` | Maps timestamp index (0 = oldest) to a `'CV-Experiment'` label |

In [ ]:
# ---- CHANGE THESE ------------------------------------------------
# label -> (folder_name, tags)
# Run Section 4 first to see which index maps to which timestamp.
RUNS = {
    ('svm', 'selectkbest'): ('svm_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('svm', 'pca'):         ('svm_pca',         {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('svm', 'sdae'):        ('svm_sdae',         {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'selectkbest'): ('rf_selectkbest',  {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'pca'):         ('rf_pca',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('rf',  'sdae'):        ('rf_sdae',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'selectkbest'): ('xgb_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'pca'):         ('xgb_pca',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('xgb', 'sdae'):        ('xgb_sdae',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'selectkbest'): ('gcn_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'pca'):         ('gcn_pca',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('gcn', 'sdae'):        ('gcn_sdae',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
}

# Class labels: row/col order matches [[TN, FP], [FN, TP]]
CLASS_LABELS = ['TD', 'ASD']

# Filter which runs to plot.
# Set to None to include all, or a list of label strings from RUNS.
FILTER = {
    'experiments': ['Baseline'],   # None or e.g. ['Baseline', 'Optimised']
    'cv':          ['SKF'],   # None or e.g. ['SKF', 'LOSO']
    'models':      ['rf'],   # None or e.g. ['svm']
    'dims':        ['selectkbest'],   # None or e.g. ['selectkbest', 'pca']
}

## 4. Discover Timestamps **[CONFIGURE]**

Scans each logger folder and prints all discovered timestamps with their index.

In [ ]:
seen_folders = set()
for label, (folder_name, tags) in RUNS.items():
    if folder_name in seen_folders:
        continue
    seen_folders.add(folder_name)
    timestamps = discover_timestamps(folder_name)
    print(f"{folder_name}")
    if not timestamps:
        print('  (no runs found)')
    for i, ts in enumerate(timestamps):
        current_tag = tags.get(i, '(untagged)')
        print(f'  [{i}]  {ts}  ->  {current_tag}')
    print()

## 5. Load and Aggregate

Resolves timestamps from the `tags` in `RUNS`, applies `FILTER`, and aggregates confusion matrices across all folds or sites for each run. Prints sensitivity, specificity, and accuracy for each.

In [ ]:
VALID_TAGS = {'SKF-Baseline', 'SKF-Optimised', 'LOSO-Baseline', 'LOSO-Optimised'}

# Resolve FILTER
active_experiments = FILTER['experiments'] or ['Baseline', 'Optimised']
active_cv          = FILTER['cv']          or ['SKF', 'LOSO']
active_models      = FILTER['models']      or list({m for m, _ in RUNS})
active_dims        = FILTER['dims']        or list({d for _, d in RUNS})

# aggregated_cms[(model, dim, cv, experiment)] = 2x2 np.ndarray
aggregated_cms = {}

for (model, dim), (folder_name, tags) in RUNS.items():
    if model not in active_models or dim not in active_dims:
        continue

    timestamps = discover_timestamps(folder_name)

    for ts_idx, tag in tags.items():
        if tag not in VALID_TAGS:
            print(f"WARNING: unknown tag '{tag}' for {folder_name}[{ts_idx}] skipping")
            continue

        cv, exp = tag.split('-')

        if cv not in active_cv or exp not in active_experiments:
            continue

        if ts_idx >= len(timestamps):
            print(f"WARNING: index {ts_idx} out of range for {folder_name} "
                  f"({len(timestamps)} runs found) skipping")
            continue

        timestamp     = timestamps[ts_idx]
        logger        = CentralisedLogger(model_name=folder_name)
        target_folder = logger.artefact_path / timestamp

        if not target_folder.exists():
            print(f"WARNING: folder not found {target_folder}")
            continue

        raw = logger.load_folder_artefacts(folder_path=target_folder)

        agg = np.zeros((2, 2), dtype=int)
        for entry in raw.values():
            agg += np.array(entry['cm'], dtype=int)

        tn, fp, fn, tp = agg[0, 0], agg[0, 1], agg[1, 0], agg[1, 1]
        total = agg.sum()
        sens  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        spec  = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
        acc   = (tp + tn) / total if total > 0 else float('nan')

        label    = f"{model.upper()} + {dim}"
        cv_label = 'Stratified 5-Fold CV' if cv == 'SKF' else 'Leave-One-Site-Out CV'
        print(f"{label}  |  {cv_label}  |  {exp}")
        print(f"  Folds/sites loaded : {len(raw)}")
        print(f"  Aggregated CM      : {agg.tolist()}")
        print(f"  Sensitivity        : {sens:.3f}")
        print(f"  Specificity        : {spec:.3f}")
        print(f"  Accuracy           : {acc:.3f}")
        print()

        aggregated_cms[(model, dim, cv, exp)] = agg

## 6. Plot Settings

Controls figure titles and output filenames. Titles are generated automatically from the run label, CV scheme, and experiment type. Edit `TITLE_MAP` or `FILENAME_MAP` here to override for specific runs.

In [ ]:
# Optional overrides. Leave empty to use defaults
TITLE_MAP    = {}   # e.g. {('RF + PCA + LOSO', 'LOSO', 'Optimised'): 'Custom title'}
FILENAME_MAP = {}   # e.g. {('RF + PCA + LOSO', 'LOSO', 'Optimised'): 'my_cm.pdf'}

## 7. Generate Figures

Plots a normalised confusion matrix for each loaded run. Each cell shows the row-normalised proportion and the raw count in parentheses.

In [ ]:
for (model, dim, cv, exp), agg_cm in aggregated_cms.items():
    label     = f"{model.upper()} + {dim}"
    title     = TITLE_MAP.get((model, dim, cv, exp),     default_title(label, cv, exp))
    plot_confusion_matrix(cm=agg_cm, title=title, class_labels=CLASS_LABELS, save=True)